# Step 1: What Are We Building?

A complete **N-Layer Neural Network from scratch** using only:

```python
import numpy as np
import pandas as pd
```

No TensorFlow.  
No PyTorch.  
No Scikit-Learn.

---

## Network Architecture

```text
Input Layer (2)
      ↓
Hidden Layer (2)
      ↓
Leaky ReLU
      ↓
Output Layer (3)
      ↓
Softmax
```

### Inputs
- Sweetness
- Size

### Outputs
- Apple
- Banana
- Orange

---

## Goal

Build every component ourselves:

1. Weight Initialization
2. Forward Propagation
3. Leaky ReLU Activation
4. Softmax Output
5. Cross-Entropy Loss
6. Backpropagation
7. Gradient Descent
8. Training Loop
9. Prediction

By the end, we'll have a fully working neural network implemented entirely with NumPy.

In [1]:
import numpy as np
import pandas as pd

X = np.array([
    [5,2],
    [2,5],
    [6,1],
    [1,6]
])

# Apple Banana Orange

y = np.array([
    [0,0,1],
    [0,1,0],
    [1,0,0],
    [0,1,0]
])

In [2]:
print(X.shape)

print(y.shape)

(4, 2)
(4, 3)


In [3]:
# Initialize Parameters
np.random.seed(42)

W1 = np.random.randn(2,2)*0.01
b1 = np.zeros((1,2))

W2 = np.random.randn(2,3)*0.01
b2 = np.zeros((1,3))

W1 = (2,2)

b1 = (1,2)

W2 = (2,3)

b2 = (1,3)

In [4]:
def leaky_relu(z):

    return np.where(z > 0,
                    z,
                    0.01*z)

In [5]:
def leaky_relu_derivative(z):

    return np.where(z > 0,
                    1,
                    0.01)

In [6]:
def softmax(z):

    z = z - np.max(z,
                   axis=1,
                   keepdims=True)

    exp_z = np.exp(z)

    return exp_z / np.sum(
        exp_z,
        axis=1,
        keepdims=True
    )

In [7]:
def cross_entropy(y_true,
                  y_pred):

    m = y_true.shape[0]

    loss = -np.sum(
        y_true*np.log(y_pred+1e-10)
    )/m

    return loss

In [8]:
def forward(X,
            W1,
            b1,
            W2,
            b2):

    Z1 = X @ W1 + b1

    A1 = leaky_relu(Z1)

    Z2 = A1 @ W2 + b2

    A2 = softmax(Z2)

    cache = (
        Z1,
        A1,
        Z2,
        A2
    )

    return A2, cache

## Computational Graph

```text
X
 ↓
Z1 = XW1 + b1
 ↓
A1 = LeakyReLU(Z1)
 ↓
Z2 = A1W2 + b2
 ↓
A2 = Softmax(Z2)
 ↓
Loss = CrossEntropy(A2, Y)
```

### Cache Everything

During forward propagation we store:

```python
cache = {
    "X": X,
    "Z1": Z1,
    "A1": A1,
    "Z2": Z2,
    "A2": A2
}
```

### Why?

Backpropagation needs these intermediate values to compute gradients.

Example:

```text
dL/dW2 requires A1
dL/dZ2 requires A2
dL/dW1 requires X
dL/dZ1 requires Z1
```

Without the cache, we'd have to recompute the entire forward pass.

**Forward Pass → Store Values → Backward Pass Uses Them**

In [9]:
def backward(X,
             y,
             cache,
             W2):

    Z1,A1,Z2,A2 = cache

    m = X.shape[0]

    dZ2 = A2 - y

    dW2 = A1.T @ dZ2 / m

    db2 = np.sum(
        dZ2,
        axis=0,
        keepdims=True
    )/m

    dA1 = dZ2 @ W2.T

    dZ1 = dA1 * leaky_relu_derivative(Z1)

    dW1 = X.T @ dZ1 / m

    db1 = np.sum(
        dZ1,
        axis=0,
        keepdims=True
    )/m

    return (
        dW1,
        db1,
        dW2,
        db2
    )

## Backpropagation = Chain Rule

During the backward pass, gradients flow in the reverse direction:

```text
Loss
 ↑
A2
 ↑
Z2
 ↑
A1
 ↑
Z1
 ↑
W1
```

The gradient at each step is computed using the **Chain Rule**:

```text
dL/dW1
=
(dL/dA2)
× (dA2/dZ2)
× (dZ2/dA1)
× (dA1/dZ1)
× (dZ1/dW1)
```

For our network:

```text
Loss
 ↓
Softmax + CrossEntropy
 ↓
Output Layer
 ↓
Leaky ReLU
 ↓
Hidden Layer
```

Backpropagation is simply:

1. Start with the loss gradient.
2. Move layer by layer backward.
3. Apply the chain rule.
4. Compute gradients for weights and biases.
5. Update parameters using Gradient Descent.

### Key Idea

```text
Forward Pass  → Compute Values
Backward Pass → Compute Gradients
```

Backpropagation is nothing more than the **Chain Rule implemented with matrix operations in NumPy**.

In [10]:
def update_params(
        W1,b1,
        W2,b2,
        dW1,db1,
        dW2,db2,
        lr):

    W1 -= lr*dW1
    b1 -= lr*db1

    W2 -= lr*dW2
    b2 -= lr*db2

    return W1,b1,W2,b2

In [12]:
# Training Loop
epochs = 5000

lr = 0.1

for epoch in range(epochs):

    A2,cache = forward(
        X,
        W1,b1,
        W2,b2
    )

    loss = cross_entropy(
        y,
        A2
    )

    dW1,db1,dW2,db2 = backward(
        X,
        y,
        cache,
        W2
    )

    W1,b1,W2,b2 = update_params(
        W1,b1,
        W2,b2,
        dW1,db1,
        dW2,db2,
        lr
    )

    if epoch % 500 == 0:

        print(
            epoch,
            loss
        )

0 0.0002526931351100808
500 0.00022530827892371005
1000 0.0002030264248626904
1500 0.00018455309948103182
2000 0.00016900552640353487
2500 0.000155751640670765
3000 0.00014432779916726378
3500 0.00013438620261900335
4000 0.00012566103644972535
4500 0.00011795365249108196


In [13]:
# Prediction
A2,_ = forward(
    X,
    W1,b1,
    W2,b2
)

preds = np.argmax(
    A2,
    axis=1
)

print(preds)

[2 1 0 1]
